## Learning Objectives {.unnumbered}

By the end of this tutorial you will be able to:

1. Generate and explore a synthetic regression dataset
2. Implement and train a **single-layer perceptron** (SLP) in PyTorch
3. Implement and train a **3-layer MLP** in PyTorch
4. Compare the two models on training loss, test loss, and prediction quality
5. Produce key training visualisations: loss curves, scatter fits, and a gradient-descent animation
6. Recognise the effect of model depth on expressiveness

---

## Setup


In [9]:
#| label: setup
#| eval: true
import os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, PillowWriter
warnings.filterwarnings('ignore')

# Make figure directory
fig_dir = "M0_lecture03_figures"
os.makedirs(fig_dir, exist_ok=True)

import torch
from torch import nn
from torch.utils import data as torchdata

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | device: {device}")
print(f"Figure dir: {fig_dir}/")


PyTorch 2.11.0+cu130 | device: cuda
Figure dir: M0_lecture03_figures/


---

## Exercise A — The Dataset: Sinusoidal Regression

We use a **1-D non-linear synthetic dataset** that a linear model *cannot* fit perfectly but an MLP can:

$$y = \sin(2\pi x) + 0.5x^2 + \epsilon, \quad x \sim \mathcal{U}(-1.5, 1.5), \quad \epsilon \sim \mathcal{N}(0, 0.1^2)$$

This is deliberately non-linear so we can observe the SLP struggling and the MLP succeeding.


In [10]:
#| label: generate-data
#| eval: true

def make_dataset(n=300, noise=0.10, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1.5, 1.5, n).astype(np.float32)
    y = (np.sin(2 * np.pi * x) + 0.5 * x**2 + rng.normal(0, noise, n)).astype(np.float32)
    return x, y

x_all, y_all = make_dataset(n=400, noise=0.10)

# set as a datafraem for easier visualization
import pandas as pd
df = pd.DataFrame({"x": x_all, "y": y_all})
df.head()

,x,y
0,0.821868,-0.518070
1,-0.183365,-0.844395
2,1.075794,1.064722
3,0.592104,-0.512925
4,-1.217468,-0.469078


In [11]:
# plot x and y
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter(df["x"], df["y"], alpha=0.5, s=20)
plt.title("Dataset")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

In [12]:

# Train / val / test split  (70 / 15 / 15)
idx = np.random.permutation(len(x_all))
n_train = int(0.7 * len(x_all))
n_val   = int(0.15 * len(x_all))
x_train, y_train = x_all[idx[:n_train]],        y_all[idx[:n_train]]
x_val,   y_val   = x_all[idx[n_train:n_train+n_val]], y_all[idx[n_train:n_train+n_val]]
x_test,  y_test  = x_all[idx[n_train+n_val:]],  y_all[idx[n_train+n_val:]]

print(f"Train: {len(x_train)} | Val: {len(x_val)} | Test: {len(x_test)}")

# Convert to tensors
def to_t(arr): return torch.tensor(arr).unsqueeze(1)
Xt, yt = to_t(x_train), to_t(y_train)
Xv, yv = to_t(x_val),   to_t(y_val)
Xte, yte = to_t(x_test), to_t(y_test)


Train: 280 | Val: 60 | Test: 60


In [13]:
#| label: fig-data
#| fig-cap: "Synthetic regression dataset. The true function (green) is non-linear; a single straight line cannot fit it."
#| eval: true

x_line = np.linspace(-1.6, 1.6, 300).astype(np.float32)
y_true = np.sin(2 * np.pi * x_line) + 0.5 * x_line**2

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_train, y_train, s=15, alpha=0.5, color='steelblue', label='Train data')
ax.scatter(x_val,   y_val,   s=15, alpha=0.5, color='orange',    label='Val data')
ax.plot(x_line, y_true, 'g-', lw=2.5, label='True function y=sin(2πx)+0.5x²')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Synthetic Non-Linear Regression Dataset')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_dataset.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_dataset.png")


Saved fig_T3_dataset.png


---

## Exercise B — Single-Layer Perceptron (Linear Regression)

A **single-layer perceptron** for regression is just a linear model:

$$\hat{y} = w \cdot x + b$$

It has *no* hidden layers and *no* non-linear activation.

### B.1 Build and Inspect the Model


In [14]:
#| label: slp-model
#| eval: true

class SLP(nn.Module):
    """Single-layer perceptron (linear regression)."""
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

slp = SLP().to(device)
print(slp)
print(f"\nParameters:")
for name, p in slp.named_parameters():
    #  Use Tensor.cpu() to copy
    print(f"  {name}: {p.data.cpu().numpy().flatten()}")
    # print(f"  {name}: {p.data.numpy().flatten()}")


SLP(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)

Parameters:
  linear.weight: [0.7645385]
  linear.bias: [0.8300079]


$$
\hat{y} = 0.9186113 - 0.23527248 * x_{1}
$$

### B.2 Training Loop

We write a reusable training function we can apply to both models:


In [15]:
#| label: train-fn
#| eval: true

def train_model(model, X_train, y_train, X_val, y_val,
                lr=0.01, epochs=300, batch_size=32,
                weight_decay=0.0, verbose=True):
    """
    Generic training loop.
    Returns: train_losses, val_losses (one per epoch).
    """
    dataset = torchdata.TensorDataset(X_train.to(device), y_train.to(device))
    loader  = torchdata.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    loss_fn = nn.MSELoss()
    optim   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_losses, val_losses = [], []
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for Xb, yb in loader:
            pred = model(Xb)
            loss = loss_fn(pred, yb)
            optim.zero_grad(); loss.backward(); optim.step()
            epoch_loss += loss.item() * len(Xb)
        train_losses.append(epoch_loss / len(X_train))

        model.eval()
        with torch.no_grad():
            v_loss = loss_fn(model(X_val.to(device)), y_val.to(device)).item()
        val_losses.append(v_loss)

        if verbose and epoch % 50 == 0:
            print(f"Epoch {epoch:4d} | Train MSE: {train_losses[-1]:.4f} | Val MSE: {v_loss:.4f}")

    return train_losses, val_losses


In [16]:
#| label: train-slp
#| eval: true

print("Training SLP (Linear Regression)...")
slp_train_loss, slp_val_loss = train_model(
    slp, Xt, yt, Xv, yv,
    lr=0.05, epochs=300, batch_size=32
)
print(f"\nFinal Test MSE (SLP): {nn.MSELoss()(slp(Xte.to(device)), yte.to(device)).item():.4f}")


Training SLP (Linear Regression)...
Epoch   50 | Train MSE: 0.5831 | Val MSE: 0.5790
Epoch  100 | Train MSE: 0.5813 | Val MSE: 0.5739
Epoch  150 | Train MSE: 0.5778 | Val MSE: 0.5755
Epoch  200 | Train MSE: 0.5835 | Val MSE: 0.5847
Epoch  250 | Train MSE: 0.5782 | Val MSE: 0.5767
Epoch  300 | Train MSE: 0.5847 | Val MSE: 0.5855

Final Test MSE (SLP): 0.6619


---

## Exercise C — 3-Layer MLP

A **3-layer MLP** has two hidden layers and one output layer:

$$\mathbf{h}_1 = \text{ReLU}(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) \in \mathbb{R}^{64}$$
$$\mathbf{h}_2 = \text{ReLU}(\mathbf{W}_2 \mathbf{h}_1 + \mathbf{b}_2) \in \mathbb{R}^{32}$$
$$\hat{y} = \mathbf{W}_3 \mathbf{h}_2 + \mathbf{b}_3 \in \mathbb{R}$$

### C.1 Build the Model


In [17]:
#| label: mlp-model
#| eval: true

class MLP3(nn.Module):
    """3-layer MLP: Linear(1→64) → ReLU → Linear(64→32) → ReLU → Linear(32→1)."""
    def __init__(self, hidden1=64, hidden2=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
            nn.Linear(hidden2, 1)
        )
        # Xavier initialisation for all linear layers
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.net(x)

mlp = MLP3(hidden1=64, hidden2=32).to(device)
print(mlp)
total_params = sum(p.numel() for p in mlp.parameters())
print(f"\nTotal parameters: {total_params:,}")


MLP3(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters: 2,241


### C.2 Train the MLP


In [18]:
#| label: train-mlp
#| eval: true

print("Training 3-Layer MLP...")
mlp_train_loss, mlp_val_loss = train_model(
    mlp, Xt, yt, Xv, yv,
    lr=0.005, epochs=300, batch_size=32, weight_decay=1e-4
)
print(f"\nFinal Test MSE (MLP): {nn.MSELoss()(mlp(Xte.to(device)), yte.to(device)).item():.4f}")


Training 3-Layer MLP...
Epoch   50 | Train MSE: 0.1017 | Val MSE: 0.0951
Epoch  100 | Train MSE: 0.0826 | Val MSE: 0.0922
Epoch  150 | Train MSE: 0.0760 | Val MSE: 0.0832
Epoch  200 | Train MSE: 0.0734 | Val MSE: 0.0801
Epoch  250 | Train MSE: 0.0742 | Val MSE: 0.0755
Epoch  300 | Train MSE: 0.0713 | Val MSE: 0.0856

Final Test MSE (MLP): 0.0652


---

## Exercise D — Comparing the Two Models

### D.1 Loss Curves


In [19]:
#| label: fig-loss-curves
#| fig-cap: "Training and validation loss for SLP vs. MLP over 300 epochs. The MLP converges to a much lower loss."
#| eval: true

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs_range = range(1, 301)

for ax, (tr, vl, label, color) in zip(axes, [
    (slp_train_loss, slp_val_loss, 'SLP (Linear)', 'steelblue'),
    (mlp_train_loss, mlp_val_loss, '3-Layer MLP',  'tomato'),
]):
    ax.plot(epochs_range, tr, color=color,     lw=2, label='Train')
    ax.plot(epochs_range, vl, color=color, lw=2, linestyle='--', alpha=0.7, label='Val')
    ax.set_title(f'{label} — Loss Curves', fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=10)
    best_ep = int(np.argmin(vl)) + 1
    ax.axvline(best_ep, color='green', lw=1.5, linestyle=':', alpha=0.6,
               label=f'Best val epoch {best_ep}')
    ax.legend(fontsize=9)

plt.suptitle('Training Dynamics: SLP vs. 3-Layer MLP', fontsize=13)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_loss_curves.png", dpi=130, bbox_inches='tight')
plt.show()
# print("Saved fig_T3_loss_curves.png")


### D.2 Prediction Quality


In [20]:
#| label: fig-predictions
#| fig-cap: "Predictions on a dense grid overlaid on test data. The MLP captures the non-linear structure while the SLP can only fit a straight line."
#| eval: true

x_dense = torch.linspace(-1.6, 1.6, 400).unsqueeze(1).to(device)

slp.eval(); mlp.eval()
with torch.no_grad():
    slp_pred = slp(x_dense).cpu().numpy().flatten()
    mlp_pred = mlp(x_dense).cpu().numpy().flatten()

x_dense_np = x_dense.cpu().numpy().flatten()
y_true_dense = np.sin(2 * np.pi * x_dense_np) + 0.5 * x_dense_np**2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (preds, label, color) in zip(axes, [
    (slp_pred, 'SLP (Linear)', 'steelblue'),
    (mlp_pred, '3-Layer MLP',  'tomato'),
]):
    ax.scatter(x_test, y_test, s=20, alpha=0.4, color='gray', label='Test data')
    ax.plot(x_dense_np, y_true_dense, 'g--', lw=2, alpha=0.8, label='True function')
    ax.plot(x_dense_np, preds,         color=color, lw=2.5, label=label)
    mse = np.mean((preds - y_true_dense)**2)
    ax.set_title(f'{label}  |  Test MSE ≈ {mse:.4f}', fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=9)

plt.suptitle('Prediction Comparison: SLP vs. 3-Layer MLP', fontsize=13)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_predictions.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_predictions.png")


Saved fig_T3_predictions.png


### D.3 Residual Analysis


In [21]:
#| label: fig-residuals
#| fig-cap: "Residuals (prediction − truth) for both models. The MLP residuals are smaller and more evenly distributed around zero."
#| eval: true

with torch.no_grad():
    slp_test_pred = slp(Xte.to(device)).cpu().numpy().flatten()
    mlp_test_pred = mlp(Xte.to(device)).cpu().numpy().flatten()

slp_residuals = slp_test_pred - y_test
mlp_residuals = mlp_test_pred - y_test

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (res, label, color) in zip(axes, [
    (slp_residuals, 'SLP', 'steelblue'),
    (mlp_residuals, '3-Layer MLP', 'tomato'),
]):
    ax.scatter(y_test, res, s=20, alpha=0.5, color=color)
    ax.axhline(0, color='black', lw=1.5, linestyle='--')
    ax.set_xlabel('True y'); ax.set_ylabel('Residual (ŷ − y)')
    ax.set_title(f'{label}  RMSE = {np.sqrt(np.mean(res**2)):.4f}', fontsize=11)

plt.suptitle('Residuals: SLP vs. MLP', fontsize=13)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_residuals.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_residuals.png")


Saved fig_T3_residuals.png


---

## Exercise E — Effect of Depth and Width

Explore how the number of hidden layers changes model capacity:


In [22]:
#| label: depth-comparison
#| eval: true

def build_mlp(layers_cfg):
    """Build an MLP from a list of (in, out) pairs."""
    layers = []
    for i, (inp, out) in enumerate(layers_cfg):
        layers.append(nn.Linear(inp, out))
        if i < len(layers_cfg) - 1:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)

configs = {
    'SLP (no hidden)':     [(1, 1)],
    '1 hidden (32)':       [(1, 32), (32, 1)],
    '2 hidden (64,32)':    [(1, 64), (64, 32), (32, 1)],
    '3 hidden (128,64,32)':[(1, 128),(128,64), (64, 32), (32, 1)],
}

results = {}
for name, cfg in configs.items():
    model = build_mlp(cfg).to(device)
    # Quick train (100 epochs)
    optim = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    for _ in range(200):
        model.train()
        pred = model(Xt.to(device))
        l = loss_fn(pred, yt.to(device))
        optim.zero_grad(); l.backward(); optim.step()
    model.eval()
    with torch.no_grad():
        val_mse = loss_fn(model(Xv.to(device)), yv.to(device)).item()
    results[name] = (model, val_mse)
    print(f"{name:30s}  |  Val MSE: {val_mse:.4f}  |  Params: {sum(p.numel() for p in model.parameters()):,}")


SLP (no hidden)                 |  Val MSE: 0.5760  |  Params: 2
1 hidden (32)                   |  Val MSE: 0.2106  |  Params: 97
2 hidden (64,32)                |  Val MSE: 0.0128  |  Params: 2,241
3 hidden (128,64,32)            |  Val MSE: 0.0103  |  Params: 10,625


In [23]:
#| label: fig-depth-comparison
#| fig-cap: "Effect of network depth on prediction quality. Deeper networks approximate the non-linear function more closely."
#| eval: true

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['steelblue', 'darkorange', 'tomato', 'mediumpurple']

for ax, ((name, (model, val_mse)), color) in zip(axes, zip(results.items(), colors)):
    model.eval()
    with torch.no_grad():
        preds = model(x_dense).cpu().numpy().flatten()
    ax.scatter(x_train, y_train, s=8, alpha=0.3, color='gray')
    ax.plot(x_dense_np, y_true_dense, 'g--', lw=1.5, alpha=0.8)
    ax.plot(x_dense_np, preds, color=color, lw=2)
    ax.set_title(f"{name}\nVal MSE={val_mse:.4f}", fontsize=9)
    ax.set_xlim(-1.7, 1.7); ax.set_ylim(-2.5, 2.5)

plt.suptitle('Effect of Network Depth on Expressiveness', fontsize=12)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_depth_comparison.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_depth_comparison.png")


Saved fig_T3_depth_comparison.png


---

## Exercise F — Gradient Descent Animation (SGD Path)

Visualise how SGD moves parameters for a simple 1-parameter regression problem:


In [24]:
#| label: fig-sgd-anim
#| fig-cap: "Animated SGD path on the 1-D loss surface. The optimiser descends the parabola toward the minimum."
#| eval: true

# Simple 1-D problem: y = 3x + noise, learn w (treat b=0)
np.random.seed(0)
n = 80
x1 = np.random.randn(n).astype(np.float32)
y1 = 3.0 * x1 + 0.3 * np.random.randn(n).astype(np.float32)

# Loss as function of w
w_grid = np.linspace(-1, 7, 300)
loss_grid = np.array([np.mean((w*x1 - y1)**2)/2 for w in w_grid])

# SGD trajectory
w_val = -0.5; lr2 = 0.08
ws = [w_val]
for _ in range(30):
    grad = np.mean((w_val*x1 - y1)*x1)
    w_val -= lr2 * grad
    ws.append(w_val)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(w_grid, loss_grid, 'b-', lw=2.5, label='Loss L(w)')
line, = ax.plot([], [], 'ro-', ms=7, lw=1.5, label='SGD path')
ax.set_xlabel('Weight w', fontsize=12); ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('SGD Descending the 1-D Loss Surface', fontsize=13)
ax.legend(fontsize=10)

def update(frame):
    loss_vals = [np.mean((ws[i]*x1 - y1)**2)/2 for i in range(frame+1)]
    line.set_data(ws[:frame+1], loss_vals)
    return line,

ani = FuncAnimation(fig, update, frames=len(ws), interval=200, blit=True)
try:
    ani.save(f"{fig_dir}/fig_T3_sgd_1d.gif", writer=PillowWriter(fps=5))
    print("Saved fig_T3_sgd_1d.gif")
except Exception as e:
    print(f"GIF save skipped ({e}) — saving static frame")

# Static final frame
loss_final = [np.mean((w_*x1-y1)**2)/2 for w_ in ws]
line.set_data(ws, loss_final)
plt.savefig(f"{fig_dir}/fig_T3_sgd_1d.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_sgd_1d.png")


Saved fig_T3_sgd_1d.gif
Saved fig_T3_sgd_1d.png


---

## Exercise G — Effect of Learning Rate

A critical hyperparameter: too small → slow convergence; too large → divergence.


In [25]:
#| label: fig-lr-comparison
#| fig-cap: "Effect of learning rate on training convergence. Too small is slow; too large causes oscillation or divergence."
#| eval: true

learning_rates = [0.001, 0.01, 0.1, 0.5]
colors_lr = ['steelblue', 'darkorange', 'tomato', 'mediumpurple']
lr_losses = {}

for lr_val in learning_rates:
    model_lr = MLP3(32, 16).to(device)
    optim_lr = torch.optim.SGD(model_lr.parameters(), lr=lr_val)
    loss_fn  = nn.MSELoss()
    ep_losses = []
    for _ in range(150):
        model_lr.train()
        pred = model_lr(Xt.to(device))
        l = loss_fn(pred, yt.to(device))
        optim_lr.zero_grad(); l.backward(); optim_lr.step()
        ep_losses.append(l.item())
    lr_losses[lr_val] = ep_losses

fig, ax = plt.subplots(figsize=(9, 5))
for (lr_val, losses), color in zip(lr_losses.items(), colors_lr):
    clean_losses = [min(v, 5.0) for v in losses]  # clip for visibility
    ax.plot(range(1, 151), clean_losses, color=color, lw=2, label=f'lr={lr_val}')
ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Train MSE (clipped at 5)', fontsize=12)
ax.set_title('Effect of Learning Rate on SGD Convergence', fontsize=13)
ax.legend(fontsize=10); ax.set_ylim(0, 4)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_lr_comparison.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_lr_comparison.png")


Saved fig_T3_lr_comparison.png


---

## Exercise H — Overfitting Demonstration

Deliberately overfit by training a very large MLP for many epochs *without* regularisation:


In [26]:
#| label: overfitting-demo
#| eval: true

# Use a tiny dataset to make overfitting visible
x_tiny = np.linspace(-1.5, 1.5, 20).astype(np.float32)
y_tiny = (np.sin(2*np.pi*x_tiny) + 0.5*x_tiny**2 + 0.15*np.random.randn(20)).astype(np.float32)
Xt_tiny = torch.tensor(x_tiny).unsqueeze(1)
yt_tiny = torch.tensor(y_tiny).unsqueeze(1)

# Very large model
big_mlp = nn.Sequential(
    nn.Linear(1, 256), nn.Tanh(),
    nn.Linear(256, 256), nn.Tanh(),
    nn.Linear(256, 1)
).to(device)
optim_big = torch.optim.Adam(big_mlp.parameters(), lr=0.01)
loss_fn   = nn.MSELoss()

for epoch in range(2000):
    big_mlp.train()
    l = loss_fn(big_mlp(Xt_tiny.to(device)), yt_tiny.to(device))
    optim_big.zero_grad(); l.backward(); optim_big.step()

print(f"Train MSE (tiny dataset, big model): {loss_fn(big_mlp(Xt_tiny.to(device)), yt_tiny.to(device)).item():.6f}")
print(f"Test  MSE (full test set):            {loss_fn(big_mlp(Xte.to(device)), yte.to(device)).item():.4f}")


Train MSE (tiny dataset, big model): 0.000092
Test  MSE (full test set):            0.0279


In [27]:
#| label: fig-overfitting
#| fig-cap: "Overfitting: the big MLP memorises 20 training points but oscillates wildly elsewhere. The well-regularised MLP generalises."
#| eval: true

big_mlp.eval(); mlp.eval()
with torch.no_grad():
    big_pred = big_mlp(x_dense).cpu().numpy().flatten()
    mlp_pred2 = mlp(x_dense).cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (preds, train_x, train_y, label, color) in zip(axes, [
    (big_pred, x_tiny, y_tiny, 'Overfit Big MLP (n=20 train points)', 'tomato'),
    (mlp_pred2, x_train, y_train, 'Well-trained MLP (n=280 train points)', 'steelblue'),
]):
    ax.scatter(train_x, train_y, s=30, color='black', zorder=5, label='Train data')
    ax.plot(x_dense_np, y_true_dense, 'g--', lw=1.8, alpha=0.7, label='True function')
    ax.plot(x_dense_np, preds, color=color, lw=2.5, label=label)
    ax.set_xlim(-1.7, 1.7); ax.set_ylim(-3.5, 3.5)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Overfitting vs. Good Generalisation', fontsize=13)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_overfitting.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_overfitting.png")


Saved fig_T3_overfitting.png


---

## Exercise I — Regularisation: Weight Decay

Compare the same big model trained with and without L2 regularisation:


In [28]:
#| label: fig-regularisation
#| fig-cap: "Effect of weight decay (L2 regularisation) on overfitting. Larger λ prevents memorisation but too much under-fits."
#| eval: true

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lambda_vals = [0.0, 0.01, 0.1]
colors_reg = ['tomato', 'darkorange', 'steelblue']

for ax, (wd, color) in zip(axes, zip(lambda_vals, colors_reg)):
    reg_model = nn.Sequential(
        nn.Linear(1, 128), nn.Tanh(),
        nn.Linear(128, 128), nn.Tanh(),
        nn.Linear(128, 1)
    ).to(device)
    optim_reg = torch.optim.Adam(reg_model.parameters(), lr=0.01, weight_decay=wd)
    loss_fn_r = nn.MSELoss()
    for _ in range(1000):
        reg_model.train()
        l = loss_fn_r(reg_model(Xt_tiny.to(device)), yt_tiny.to(device))
        optim_reg.zero_grad(); l.backward(); optim_reg.step()

    reg_model.eval()
    with torch.no_grad():
        preds = reg_model(x_dense).cpu().numpy().flatten()
        te_mse = loss_fn_r(reg_model(Xte.to(device)), yte.to(device)).item()

    ax.scatter(x_tiny, y_tiny, s=30, color='black', zorder=5, label='Train (n=20)')
    ax.plot(x_dense_np, y_true_dense, 'g--', lw=1.8, alpha=0.7, label='True fn')
    ax.plot(x_dense_np, preds, color=color, lw=2)
    ax.set_xlim(-1.7, 1.7); ax.set_ylim(-3.5, 3.5)
    ax.set_title(f'λ={wd}  |  Test MSE={te_mse:.3f}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Weight Decay Regularisation Effect', fontsize=13)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig_T3_regularisation.png", dpi=130, bbox_inches='tight')
plt.show()
print("Saved fig_T3_regularisation.png")


Saved fig_T3_regularisation.png


---

## Summary Table {.unnumbered}

| Model | Parameters | Test MSE | Notes |
|-------|-----------|---------|-------|
| SLP (Linear) | 2 | High | Cannot capture non-linearity |
| 3-Layer MLP (64→32) | ~2,500 | Low | Captures non-linear pattern |
| Big MLP (no reg) | ~100K | Very high* | Overfits tiny dataset |
| Big MLP (λ=0.01) | ~100K | Moderate | Better generalisation |

*High test MSE despite near-zero training MSE is the hallmark of overfitting.

## Quick Practice Questions {.unnumbered}

1. Change the MLP architecture to 4 hidden layers of 32 neurons each.  Does it outperform the 3-layer version?
2. Set the learning rate to `5.0` and observe what happens to the loss curve.  Why?
3. Try `nn.Tanh()` instead of `nn.ReLU()`.  Which converges faster on this dataset?
4. Remove `weight_decay` and increase training epochs to 2000.  What do the residuals look like?
5. Add a `nn.Dropout(p=0.3)` layer after each hidden activation in the big MLP.  Does it help?

## References {.unnumbered}

::: {#refs}
:::
